
# Comparing Slip-Link Sampling Methods: Discrete vs. Analytical

In polymer network elasticity simulations using the Maximum Entropy Homogenization
Procedure (MEHP) and Force Balance (:cite:t:`gusev_numerical_2019`, :cite:t:`bernhard_phantom_2025`),
entanglements are represented as slip-links or localized crosslink constraints.

Historically, sampling entanglements required instantiating the complete bead-spring
:class:`~pylimer_tools_cpp.Universe` (:meth:`~pylimer_tools_cpp.MCUniverseGenerator.get_universe`)
with 3D coordinates for every single bead along every strand. While exact, this creates
an enormous memory bottleneck: networks with hundreds of thousands or millions of beads
require gigabytes of RAM just to store atom tables, neighbor lists, and graph structures
that are immediately discarded once the network is mapped into Hookean springs.

To eliminate this memory wall, ``pylimer-tools`` introduces an **analytical Brownian-bridge
sampling method** (:meth:`~pylimer_tools_cpp.MCUniverseGenerator.sample_entanglements` and
:func:`~pylimer_tools_cpp.randomly_sample_entanglements_analytical`). Operating directly on
the lightweight :class:`~pylimer_tools_cpp.CrosslinkerUniverse` without generating intermediate
beads, it evaluates conditional Gaussian distributions between crosslinker endpoints and
constructs :class:`~pylimer_tools_cpp.MEHPForceBalance` or :class:`~pylimer_tools_cpp.MEHPForceBalance2`
networks directly.

This example illustrates:

1. **Memory & Runtime Scaling**: The order-of-magnitude memory reduction achieved by bypassing full bead generation.
2. **Conformational Distributions**: Why analytical Brownian-bridge sampling faithfully replicates discrete spatial statistics.
3. **Network Topology & Modulus across Conversion**: How topological filtering of
   dangling/soluble fractions influences network properties and shear modulus, evaluated
   with multi-seed ensembles and 95% confidence intervals.
4. **Practical Decision Guide**: When to choose which method, key trade-offs, and physical assumptions.


## Background: The Memory Wall in Slip-Link Sampling
In a polymer network, strands between crosslinkers fluctuate thermally.
Under the affine network model and Gaussian chain approximation, the position
$\mathbf{r}(\alpha)$ of a bead at contour fraction $\alpha \in (0, 1)$
on a strand connecting crosslinker anchors $\mathbf{R}_1$ and $\mathbf{R}_2$
follows a **Brownian bridge** conditional on its endpoints:

\begin{align}\mathbf{r}(\alpha) \sim \mathcal{N}\left( (1-\alpha)\mathbf{R}_1 + \alpha\mathbf{R}_2,\;
   \alpha(1-\alpha) N b^2 / 3 \,\mathbf{I} \right)\end{align}

Where:

- $N$ is the number of bonds in the strand,
- $b$ is the statistical segment length (Kuhn length or bead distance),
- $\alpha = k / N$ is the fractional contour position of bead $k$.

The traditional **Discrete Method** explicitly generates all $N_{\text{atoms}}$ beads in
3D space and constructs spatial cell lists to find pairs within cutoff $r_{\text{cut}}$.

The new **Analytical Method** skips bead coordinates entirely. It works directly on
:class:`~pylimer_tools_cpp.CrosslinkerUniverse` ($N_{\text{xlinks}} \ll N_{\text{atoms}}$),
selecting pairs of strands and sampling continuous contour positions $\alpha_1, \alpha_2$
from their conditional Gaussian distributions. Furthermore, it incorporates **topological leaf
peeling** (``filter_dangling_and_soluble=True``), discarding candidate links that involve
dangling chains or soluble molecules.



In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
import time
from typing import Any, Dict, List, Sequence

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from scipy import stats

from pylimer_tools.io.bead_spring_parameter_provider import (
    ParameterType,
    get_parameters_for_polymer,
)
from pylimer_tools.utils.example_cache import (
    compute_ensemble_statistics,
    get_example_cache_path,
    load_or_compute_cache,
)
import pylimer_tools_cpp as pt

## Benchmark 1: Scaling Benchmark (Peak Memory & Runtime)
We measure peak resident memory (RSS) and execution time across network sizes
ranging from 200 to 4,000 strands (up to 200,000+ beads).
Subprocesses isolate memory measurements per condition to prevent accumulation.



In [ ]:
def run_scaling_benchmark(
    sizes: Sequence[int] = (200, 500, 1000, 2000, 4000, 6000),
    strand_len: int = 50,
    seed: int = 42,
) -> Dict[str, np.ndarray]:
    """Measure peak memory and runtime for discrete and analytical methods across system sizes."""
    print("Running Scaling Benchmark across system sizes...")
    params = get_parameters_for_polymer("PDMS", parameter_type=ParameterType.GAUSSIAN)
    bead_density = params.get_bead_density()
    cutoff = params.get_sampling_cutoff()
    ent_density = params.get_entanglement_density()

    worker_code = """
import sys, time, resource, json
import pylimer_tools_cpp as pt

mode = sys.argv[1]
n_strands = int(sys.argv[2])
strand_len = int(sys.argv[3])
seed = int(sys.argv[4])
bead_density = float(sys.argv[5])
cutoff = float(sys.argv[6])
ent_density = float(sys.argv[7])

n_atoms = n_strands * strand_len + int(0.5 * n_strands)
vol = n_atoms / bead_density
L = vol ** (1.0 / 3.0)
n_ent = int(ent_density * vol)

t0 = time.perf_counter()
gen = pt.MCUniverseGenerator(L, L, L)
gen.set_seed(seed)
gen.set_mean_squared_bead_distance(1.0)
gen.add_strands(n_strands, [strand_len] * n_strands)
gen.add_crosslinkers(int(0.5 * n_strands), 4)
gen.link_strands_to_conversion(0.92)

if mode == "discrete":
    u = gen.get_universe()
    fb = pt.MEHPForceBalance2(
        u,
        nr_of_entanglements_to_sample=n_ent,
        upper_sampling_cutoff=cutoff,
        entanglements_as_springs=False,
    )
    fb.validate_network()
elif mode == "analytical":
    es = gen.sample_entanglements(n_ent, cutoff, 0.0, 0, 3.0, "seed", True)
    fb = gen.get_force_balance2(es, 2, False)
    fb.validate_network()

t1 = time.perf_counter()
maxrss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
if sys.platform == "darwin":
    rss_mb = maxrss / (1024.0 * 1024.0)
else:
    rss_mb = maxrss / 1024.0

print(json.dumps({"mode": mode, "n_strands": n_strands, "n_atoms": n_atoms, "time": t1 - t0, "rss_mb": rss_mb}))
"""

    disc_atoms: List[int] = []
    disc_time: List[float] = []
    disc_rss: List[float] = []

    ana_atoms: List[int] = []
    ana_time: List[float] = []
    ana_rss: List[float] = []

    for n in sizes:
        for mode in ["analytical", "discrete"]:
            cmd = [
                sys.executable,
                "-c",
                worker_code,
                mode,
                str(n),
                str(strand_len),
                str(seed),
                str(bead_density),
                str(cutoff),
                str(ent_density),
            ]
            res = subprocess.run(cmd, capture_output=True, text=True, check=True)
            lines = [l for l in res.stdout.strip().split("\n") if l.startswith("{")]
            record = json.loads(lines[-1])
            if mode == "discrete":
                disc_atoms.append(record["n_atoms"])
                disc_time.append(record["time"])
                disc_rss.append(record["rss_mb"])
                print(f"Discrete:   N_atoms={record['n_atoms']:6d} | Time={record['time']:6.2f}s | RSS={record['rss_mb']:6.1f} MB")
            else:
                ana_atoms.append(record["n_atoms"])
                ana_time.append(record["time"])
                ana_rss.append(record["rss_mb"])
                print(f"Analytical: N_atoms={record['n_atoms']:6d} | Time={record['time']:6.2f}s | RSS={record['rss_mb']:6.1f} MB")

    return {
        "scaling_atoms": np.array(ana_atoms),
        "scaling_time_discrete": np.array(disc_time),
        "scaling_rss_discrete": np.array(disc_rss),
        "scaling_time_analytical": np.array(ana_time),
        "scaling_rss_analytical": np.array(ana_rss),
    }

## Benchmark 2: Conformational Statistics Comparison
We sample entanglements and compare the distribution of continuous contour positions
$\alpha$ and spatial Euclidean distances $r$ against explicit discrete bead sampling.



In [ ]:
def run_distribution_comparison(
    n_strands: int = 4000,
    strand_len: int = 50,
    conversion: float = 0.95,
    seed: int = 123,
) -> Dict[str, np.ndarray]:
    """Sample entanglements and compare contour position alpha and spatial distance distributions."""
    print("Running Conformational Distribution Comparison...")
    params = get_parameters_for_polymer("PDMS", parameter_type=ParameterType.GAUSSIAN)
    b = params.get("<b>").to(params.get("distance_units")).magnitude
    bead_density = params.get_bead_density()
    cutoff = params.get_sampling_cutoff()
    ent_density = params.get_entanglement_density()

    n_atoms = n_strands * strand_len + int(0.5 * n_strands)
    vol = n_atoms / bead_density
    L = vol ** (1.0 / 3.0)
    n_ent = int(ent_density * vol)

    gen = pt.MCUniverseGenerator(L, L, L)
    gen.set_seed(seed)
    gen.set_bead_distance(b)
    gen.set_mean_squared_bead_distance(b**2)
    gen.add_strands(n_strands, [strand_len] * n_strands)
    gen.add_crosslinkers(int(0.5 * n_strands), 4)
    gen.link_strands_to_conversion(conversion)

    # 1. Analytical sampling (Brownian bridge)
    es = gen.sample_entanglements(n_ent, cutoff, 0.0, 0, 3.0, "ana_seed", False)
    alpha_ana = np.concatenate([np.array(es.alpha_1), np.array(es.alpha_2)])

    pos1 = np.array(es.position_1).reshape(-1, 3)
    pos2 = np.array(es.position_2).reshape(-1, 3)
    diff = pos2 - pos1
    # Minimum image convention under periodic boundary conditions
    diff = diff - np.round(diff / L) * L
    dist_ana = np.linalg.norm(diff, axis=1)

    # 2. Discrete sampling on explicit Universe beads
    u = gen.get_universe()
    cu = gen.get_crosslinker_universe()
    atom_pairs = pt.randomly_sample_entanglements(u, n_ent, cutoff, 0.0, 0, 3.0, "disc_seed")

    n_xlinks = len(cu.xlink_x)
    offsets = [n_xlinks]
    for n_beads in cu.beads_in_strand:
        offsets.append(offsets[-1] + n_beads)
    offsets_arr = np.array(offsets)

    alpha_disc = []
    dist_disc = []
    box = u.get_box()

    for a1, a2 in atom_pairs.pairs_of_atoms:
        idx1 = a1 - 1
        idx2 = a2 - 1
        if idx1 >= n_xlinks:
            s1 = np.searchsorted(offsets_arr, idx1, side="right") - 1
            b1 = idx1 - offsets_arr[s1]
            alpha_disc.append((b1 + 1.0) / (strand_len + 1.0))
        if idx2 >= n_xlinks:
            s2 = np.searchsorted(offsets_arr, idx2, side="right") - 1
            b2 = idx2 - offsets_arr[s2]
            alpha_disc.append((b2 + 1.0) / (strand_len + 1.0))

        atom1 = u.get_atom(a1)
        atom2 = u.get_atom(a2)
        dist_disc.append(atom1.distance_to(atom2, box))

    print(f"Sampled {len(dist_ana)} analytical pairs, {len(dist_disc)} discrete pairs.")
    return {
        "alpha_analytical": np.array(alpha_ana),
        "dist_analytical": np.array(dist_ana),
        "alpha_discrete": np.array(alpha_disc),
        "dist_discrete": np.array(dist_disc),
    }

## Benchmark 3: Crosslinker Conversion Sweep & Multi-Seed Ensembles
We evaluate how defective networks behave across crosslinker conversion $p \in [0.75, 1.00]$.
Multi-seed repetitions provide empirical variance and Student's t 95% confidence intervals.



In [ ]:
def run_conversion_sweep(
    conversions: Sequence[float] = (0.75, 0.80, 0.85, 0.90, 0.95, 1.00),
    n_strands: int = 1200,
    strand_len: int = 50,
    n_repeats: int = 5,
    base_seed: int = 777,
) -> Dict[str, Any]:
    """Compare modulus, active fractions, and active entanglement count across conversions."""
    print("Running Conversion Sweep across multi-seed ensembles...")
    params = get_parameters_for_polymer("PDMS", parameter_type=ParameterType.GAUSSIAN)
    b = params.get("<b>").to(params.get("distance_units")).magnitude
    bead_density = params.get_bead_density()
    cutoff = params.get_sampling_cutoff()
    ent_density = params.get_entanglement_density()
    r02 = params.get("R02").to(params.get("distance_units") ** 2).magnitude
    kbt = params.get("T") * params.get("kb")
    gamma_conv = ((kbt / ((params.get("distance_units")) ** 3)).to("MPa")).magnitude

    n_atoms = n_strands * strand_len + int(0.5 * n_strands)
    vol = n_atoms / bead_density
    L = vol ** (1.0 / 3.0)
    n_ent = int(ent_density * vol)

    n_p = len(conversions)
    g_d_all = np.zeros((n_p, n_repeats))
    w_d_all = np.zeros((n_p, n_repeats))
    g_af_all = np.zeros((n_p, n_repeats))
    w_af_all = np.zeros((n_p, n_repeats))
    n_af_all = np.zeros((n_p, n_repeats))
    g_au_all = np.zeros((n_p, n_repeats))
    w_au_all = np.zeros((n_p, n_repeats))
    n_au_all = np.zeros((n_p, n_repeats))
    g_as_all = np.zeros((n_p, n_repeats))

    for r in range(n_repeats):
        seed = base_seed + 101 * r
        print(f"Repeat {r + 1}/{n_repeats} (seed={seed})...")

        for i, p in enumerate(conversions):
            gen = pt.MCUniverseGenerator(L, L, L)
            gen.set_seed(seed)
            gen.set_bead_distance(b)
            gen.set_mean_squared_bead_distance(b**2)
            gen.add_strands(n_strands, [strand_len] * n_strands)
            gen.add_crosslinkers(int(0.5 * n_strands), 4)
            gen.link_strands_to_conversion(p)

            # 1. Discrete Full Universe
            u = gen.get_universe()
            fb_disc = pt.MEHPForceBalance2(
                u,
                nr_of_entanglements_to_sample=n_ent,
                upper_sampling_cutoff=cutoff,
                entanglements_as_springs=False,
            )
            fb_disc.run_force_relaxation()
            g_d_all[i, r] = gamma_conv * fb_disc.get_gamma_factor(r02) / vol
            w_d_all[i, r] = fb_disc.get_active_weight_fraction()

            # 2. Analytical Filtered (Default)
            es_filt = gen.sample_entanglements(n_ent, cutoff, 0.0, 0, 3.0, f"seed_f_{seed}", True)
            fb_filt = gen.get_force_balance2(es_filt, 2, False)
            fb_filt.run_force_relaxation()
            g_af_all[i, r] = gamma_conv * fb_filt.get_gamma_factor(r02) / vol
            w_af_all[i, r] = fb_filt.get_active_weight_fraction()
            n_af_all[i, r] = len(es_filt)

            # 3. Analytical Unfiltered
            es_unfilt = gen.sample_entanglements(n_ent, cutoff, 0.0, 0, 3.0, f"seed_u_{seed}", False)
            fb_unfilt = gen.get_force_balance2(es_unfilt, 2, False)
            fb_unfilt.run_force_relaxation()
            g_au_all[i, r] = gamma_conv * fb_unfilt.get_gamma_factor(r02) / vol
            w_au_all[i, r] = fb_unfilt.get_active_weight_fraction()
            n_au_all[i, r] = len(es_unfilt)

            # 4. Analytical with Hookean Springs
            fb_springs = gen.get_force_balance2(es_filt, 2, True)
            fb_springs.run_force_relaxation()
            g_as_all[i, r] = gamma_conv * fb_springs.get_gamma_factor(r02) / vol

    stat_gd = compute_ensemble_statistics(g_d_all, axis=1)
    stat_wd = compute_ensemble_statistics(w_d_all, axis=1)
    stat_gaf = compute_ensemble_statistics(g_af_all, axis=1)
    stat_waf = compute_ensemble_statistics(w_af_all, axis=1)
    stat_naf = compute_ensemble_statistics(n_af_all, axis=1)
    stat_gau = compute_ensemble_statistics(g_au_all, axis=1)
    stat_wau = compute_ensemble_statistics(w_au_all, axis=1)
    stat_nau = compute_ensemble_statistics(n_au_all, axis=1)
    stat_gas = compute_ensemble_statistics(g_as_all, axis=1)

    return {
        "conversions": np.array(conversions),
        "target_n_ent": np.array(n_ent),
        "n_repeats": np.array(n_repeats),
        "g_discrete": stat_gd["mean"],
        "g_discrete_ci": stat_gd["ci"],
        "w_active_discrete": stat_wd["mean"],
        "w_active_discrete_ci": stat_wd["ci"],
        "g_ana_filtered": stat_gaf["mean"],
        "g_ana_filtered_ci": stat_gaf["ci"],
        "w_active_ana_filtered": stat_waf["mean"],
        "w_active_ana_filtered_ci": stat_waf["ci"],
        "n_ent_ana_filtered": stat_naf["mean"],
        "n_ent_ana_filtered_ci": stat_naf["ci"],
        "g_ana_unfiltered": stat_gau["mean"],
        "g_ana_unfiltered_ci": stat_gau["ci"],
        "w_active_ana_unfiltered": stat_wau["mean"],
        "n_ent_ana_unfiltered": stat_nau["mean"],
        "g_ana_springs": stat_gas["mean"],
        "g_ana_springs_ci": stat_gas["ci"],
    }


def run_all_benchmarks() -> Dict[str, Any]:
    """Execute all benchmark suites and compile into a single dataset dictionary."""
    combined: Dict[str, Any] = {}
    combined.update(run_scaling_benchmark())
    combined.update(run_distribution_comparison())
    combined.update(run_conversion_sweep())
    return combined

## Loading Benchmark & Statistical Data
To comply with fast execution and CI limits while providing statistically
significant comparisons on large systems (up to 200,000+ beads), precomputed
benchmark data is loaded from the committed cache.

The dataset can be recomputed from scratch at any time by passing ``--recompute``
to this script or setting ``PYLIMER_RECOMPUTE_CACHE=1``.



In [ ]:
cache_path = get_example_cache_path("entanglement_sampling_comparison_cache.npz", "force_balance")
data = load_or_compute_cache(cache_path, run_all_benchmarks)

scaling_atoms = data["scaling_atoms"]
scaling_time_disc = data["scaling_time_discrete"]
scaling_rss_disc = data["scaling_rss_discrete"]
scaling_time_ana = data["scaling_time_analytical"]
scaling_rss_ana = data["scaling_rss_analytical"]

alpha_ana = data["alpha_analytical"]
dist_ana = data["dist_analytical"]
alpha_disc = data["alpha_discrete"]
dist_disc = data["dist_discrete"]

conversions = data["conversions"]
g_disc = data["g_discrete"]
w_act_disc = data["w_active_discrete"]
g_ana_filt = data["g_ana_filtered"]
w_act_ana_filt = data["w_active_ana_filtered"]
n_ent_ana_filt = data["n_ent_ana_filtered"]
g_ana_unfilt = data["g_ana_unfiltered"]
w_act_ana_unfilt = data["w_active_ana_unfiltered"]
n_ent_ana_unfilt = data["n_ent_ana_unfiltered"]
target_n_ent = int(data["target_n_ent"])

# Optional confidence interval arrays from multi-seed ensembles
g_disc_ci = data.get("g_discrete_ci")
g_ana_filt_ci = data.get("g_ana_filtered_ci")
g_ana_unfilt_ci = data.get("g_ana_unfiltered_ci")
w_act_disc_ci = data.get("w_active_discrete_ci")
w_act_ana_filt_ci = data.get("w_active_ana_filtered_ci")
n_repeats = int(data.get("n_repeats", 1))

## Comparison 1: Memory Footprint & Runtime Scaling
As the network size increases, explicit bead generation rapidly inflates peak resident
memory (RSS) due to bead coordinate arrays, neighbor search structures, and igraph graphs.
In contrast, analytical sampling bypasses intermediate bead allocation, keeping memory
footprint virtually flat and reducing peak RAM by 80% to over 90% on
large networks.



In [ ]:
fig, (ax_mem, ax_time) = plt.subplots(1, 2, figsize=(11, 4.5))


def format_k(x, pos):
    if x >= 1e6:
        return f"{x * 1e-6:.1f}".rstrip("0").rstrip(".") + "M"
    if x >= 1e3:
        return f"{int(x * 1e-3)}k"
    return str(int(x))


k_formatter = ticker.FuncFormatter(format_k)

# Peak Memory
ax_mem.plot(
    scaling_atoms,
    scaling_rss_disc,
    "o-",
    color="#d95f02",
    linewidth=2,
    markersize=6,
    label="Discrete (Full Universe)",
)
ax_mem.plot(
    scaling_atoms,
    scaling_rss_ana,
    "s-",
    color="#1b9e77",
    linewidth=2,
    markersize=6,
    label="Analytical (Universe-Free)",
)

max_atoms = scaling_atoms[-1]
mem_disc_last = float(scaling_rss_disc[-1])
mem_ana_last = float(scaling_rss_ana[-1])
pct_savings = (1.0 - mem_ana_last / mem_disc_last) * 100.0

ax_mem.annotate(
    f"{pct_savings:.0f}% Memory\nReduction",
    xy=(max_atoms, mem_ana_last),
    xytext=(max_atoms * 0.60, 75),
    arrowprops=dict(arrowstyle="->", color="#1b9e77", lw=1.5),
    fontsize=10,
    fontweight="bold",
    color="#1b9e77",
    bbox=dict(boxstyle="round,pad=0.3", fc="#e6f5f0", ec="#1b9e77", lw=1),
)

ax_mem.set_xscale("log")
ax_mem.set_yscale("log")
ax_mem.xaxis.set_major_formatter(k_formatter)
ax_mem.set_xlabel(r"Total Atoms in Network ($N_{\mathrm{atoms}}$)")
ax_mem.set_ylabel("Peak Resident Memory (RSS) [MB]")
ax_mem.set_title("Peak Memory Consumption")
ax_mem.grid(True, which="both", linestyle="--", alpha=0.5)
ax_mem.legend(frameon=True)

# Runtime
ax_time.plot(
    scaling_atoms,
    scaling_time_disc,
    "o-",
    color="#d95f02",
    linewidth=2,
    markersize=6,
    label="Discrete (Full Universe)",
)
ax_time.plot(
    scaling_atoms,
    scaling_time_ana,
    "s-",
    color="#1b9e77",
    linewidth=2,
    markersize=6,
    label="Analytical (Universe-Free)",
)

ax_time.set_xscale("log")
ax_time.set_yscale("log")
ax_time.xaxis.set_major_formatter(k_formatter)
ax_time.set_xlabel(r"Total Atoms in Network ($N_{\mathrm{atoms}}$)")
ax_time.set_ylabel("Setup & Sampling Time [s]")
ax_time.set_title("Setup & Sampling Execution Time")
ax_time.grid(True, which="both", linestyle="--", alpha=0.5)
ax_time.legend(frameon=True)

plt.tight_layout()
plt.show()

## Comparison 2: Conformational Statistics (Contour & Spatial Distributions)
A natural question is: *Does the analytical Brownian-bridge method reproduce the
same physical distribution of entanglements as explicit spatial bead sampling?*

Below, we compare:

1. **Contour position** $\alpha \in (0, 1)$: Both distributions exhibit the characteristic
   bell curve peaked near $\alpha \approx 0.5$. Beads near crosslinkers are spatially
   pinned, whereas strand midpoints explore the largest conformational volume and are most
   likely to encounter neighboring strands.
2. **Pair distance** $r \in [0, r_{\text{cut}}]$: Both methods strictly adhere to the
   sampling cutoff $r_{\text{cut}}$, producing identical Euclidean
   distance distributions.



In [ ]:
fig, (ax_alpha, ax_dist) = plt.subplots(1, 2, figsize=(11, 4.5))

# Contour fraction distribution
bins_alpha = np.linspace(0.0, 1.0, 35)
ax_alpha.hist(
    alpha_disc,
    bins=bins_alpha,
    density=True,
    alpha=0.5,
    color="#d95f02",
    label="Discrete Beads",
    edgecolor="white",
)
ax_alpha.hist(
    alpha_ana,
    bins=bins_alpha,
    density=True,
    alpha=0.5,
    color="#1b9e77",
    label="Analytical Bridge",
    edgecolor="white",
)

ax_alpha.set_xlabel(r"Contour Position $\alpha \in (0, 1)$ along Strand")
ax_alpha.set_ylabel("Probability Density")
ax_alpha.set_title("Entanglement Position along Strands")
ax_alpha.grid(True, linestyle="--", alpha=0.5)
ax_alpha.legend(frameon=True)

# Pair distance distribution
bins_dist = np.linspace(0.0, float(np.max(dist_disc)) * 1.02, 35)
ax_dist.hist(
    dist_disc,
    bins=bins_dist,
    density=True,
    alpha=0.5,
    color="#d95f02",
    label="Discrete Pairs",
    edgecolor="white",
)
ax_dist.hist(
    dist_ana,
    bins=bins_dist,
    density=True,
    alpha=0.5,
    color="#1b9e77",
    label="Analytical Pairs",
    edgecolor="white",
)

ax_dist.set_xlabel(r"Pair Separation Distance $r$ [code units]")
ax_dist.set_ylabel("Probability Density")
ax_dist.set_title(r"Entanglement Pair Separation ($r \leq r_{\mathrm{cut}}$)")
ax_dist.grid(True, linestyle="--", alpha=0.5)
ax_dist.legend(frameon=True)

plt.tight_layout()
plt.show()

## Comparison 3: Crosslinker Conversion & Topological Filtering
In real polymer networks, crosslinking reactions do not reach 100% conversion ($p < 1.0$).
This introduces topological defects: **dangling ends** (chains connected to only one crosslinker)
and **soluble fractions** (unconnected clusters).

When entanglements are sampled on defective networks, the parameter ``filter_dangling_and_soluble``
controls whether topological leaf peeling is applied before sampling:

- **Topologically Filtered Sampling** (``filter_dangling_and_soluble=True``, default):
  Iterative leaf peeling identifies dangling chains and sol. Candidate entanglement pairs
  involving dangling or sol strands are discarded, placing slip-links exclusively onto
  elastically active gel strands. This matches the default behavior of discrete sampling in
  :class:`~pylimer_tools_cpp.MEHPForceBalance2`.
- **Unfiltered Sampling** (``filter_dangling_and_soluble=False``):
  Entanglements are placed across all strands. Slip-links that land on single dangling ends
  cannot sustain macroscopic affine stress and relax to zero force.

When multi-seed ensembles are enabled, shaded confidence bands and error bars show the
**95% Student's t confidence interval** across independent realizations.



In [ ]:
fig, (ax_mod, ax_frac) = plt.subplots(1, 2, figsize=(11, 4.5))

# Convert G from MPa to Pa for clean axis display
g_disc_pa = g_disc * 1e6
g_ana_filt_pa = g_ana_filt * 1e6
g_ana_unfilt_pa = g_ana_unfilt * 1e6

# Shear modulus vs conversion
ax_mod.plot(
    conversions,
    g_disc_pa,
    "o--",
    color="#d95f02",
    linewidth=1.8,
    label="Discrete (Filtered)",
)
if g_disc_ci is not None:
    g_d_ci_pa = g_disc_ci * 1e6
    ax_mod.fill_between(
        conversions,
        g_disc_pa - g_d_ci_pa,
        g_disc_pa + g_d_ci_pa,
        color="#d95f02",
        alpha=0.15,
    )
    ax_mod.errorbar(
        conversions,
        g_disc_pa,
        yerr=g_d_ci_pa,
        fmt="none",
        ecolor="#d95f02",
        capsize=3,
        elinewidth=1.0,
    )

ax_mod.plot(
    conversions,
    g_ana_filt_pa,
    "s-",
    color="#1b9e77",
    linewidth=2.2,
    label="Analytical (Filtered, Default)",
)
if g_ana_filt_ci is not None:
    g_af_ci_pa = g_ana_filt_ci * 1e6
    ax_mod.fill_between(
        conversions,
        g_ana_filt_pa - g_af_ci_pa,
        g_ana_filt_pa + g_af_ci_pa,
        color="#1b9e77",
        alpha=0.2,
    )
    ax_mod.errorbar(
        conversions,
        g_ana_filt_pa,
        yerr=g_af_ci_pa,
        fmt="none",
        ecolor="#1b9e77",
        capsize=3,
        elinewidth=1.0,
    )

ax_mod.plot(
    conversions,
    g_ana_unfilt_pa,
    "^:",
    color="#7570b3",
    linewidth=1.8,
    label="Analytical (Unfiltered)",
)
if g_ana_unfilt_ci is not None:
    g_au_ci_pa = g_ana_unfilt_ci * 1e6
    ax_mod.fill_between(
        conversions,
        g_ana_unfilt_pa - g_au_ci_pa,
        g_ana_unfilt_pa + g_au_ci_pa,
        color="#7570b3",
        alpha=0.15,
    )
    ax_mod.errorbar(
        conversions,
        g_ana_unfilt_pa,
        yerr=g_au_ci_pa,
        fmt="none",
        ecolor="#7570b3",
        capsize=3,
        elinewidth=1.0,
    )

ax_mod.set_xlabel("Crosslinker Conversion $p$")
ax_mod.set_ylabel("Shear Modulus $G$ [Pa]")
ax_mod.set_title("Equilibrium Shear Modulus vs. Conversion")
ax_mod.grid(True, linestyle="--", alpha=0.5)
ax_mod.legend(frameon=True, fontsize=9)

# Active entanglement fraction vs conversion
frac_active_filt = n_ent_ana_filt / float(target_n_ent)

ax_frac.plot(
    conversions,
    w_act_disc,
    "o--",
    color="#d95f02",
    linewidth=1.8,
    label=r"Active Weight Fraction $w_{\mathrm{active}}$",
)
if w_act_disc_ci is not None:
    ax_frac.fill_between(
        conversions,
        w_act_disc - w_act_disc_ci,
        w_act_disc + w_act_disc_ci,
        color="#d95f02",
        alpha=0.15,
    )
    ax_frac.errorbar(
        conversions,
        w_act_disc,
        yerr=w_act_disc_ci,
        fmt="none",
        ecolor="#d95f02",
        capsize=3,
    )

ax_frac.plot(
    conversions,
    frac_active_filt,
    "s-",
    color="#1b9e77",
    linewidth=2.2,
    label=r"Filtered Active Entanglement Fraction ($N_{\mathrm{active}} / N_{\mathrm{target}}$)",
)
if w_act_ana_filt_ci is not None:
    frac_ci = w_act_ana_filt_ci
    ax_frac.fill_between(
        conversions,
        frac_active_filt - frac_ci,
        frac_active_filt + frac_ci,
        color="#1b9e77",
        alpha=0.2,
    )
    ax_frac.errorbar(
        conversions,
        frac_active_filt,
        yerr=frac_ci,
        fmt="none",
        ecolor="#1b9e77",
        capsize=3,
    )

ax_frac.set_xlabel("Crosslinker Conversion $p$")
ax_frac.set_ylabel("Fraction")
ax_frac.set_title("Network Activity & Effective Entanglements")
ax_frac.grid(True, linestyle="--", alpha=0.5)
ax_frac.legend(frameon=True, fontsize=9)

plt.tight_layout()
plt.show()

## Method Comparison & Practical Guidance
The table below summarizes key trade-offs between discrete and analytical sampling:

.. list-table::
   :widths: 22 38 40
   :header-rows: 1

   * - Criterion
     - Discrete (Full Universe)
     - Analytical (Universe-Free)
   * - **Memory Footprint**
     - High: $O(N_{\text{atoms}})$, reaches gigabytes for large networks.
     - Minimal: $O(N_{\text{xlinks}} + N_{\text{strands}})$, ~80–95% lower.
   * - **Execution Speed**
     - Slower: requires placing every bead and building 3D neighbor cell lists.
     - Fast: samples conditional Gaussian distributions between crosslinkers directly.
   * - **Strand Statistics**
     - Exact bead coordinates as generated by the Monte Carlo universe generator.
     - Assumes conditional Gaussian random walks (Brownian bridge) between anchors.
   * - **Topological Filtering**
     - Built-in leaf peeling via ``skip_dangling_soluble_entanglements=True``.
     - Built-in iterative leaf peeling via ``filter_dangling_and_soluble=True``.
   * - **Ideal Use Case**
     - Downstream MD/DPD simulations requiring full 3D atom configurations.
     - MEHP Force Balance runs, high-throughput sweeps, and large networks.

### Drawbacks and Limitations to Keep in Mind

1. **Gaussian Chain Assumption**:
   The analytical method assumes chain segments follow ideal Gaussian Brownian bridges
   between crosslinkers. For polymer melts under severe geometric confinement, highly
   stretched chains near finite extensibility (Langevin regime), or semi-flexible polymers
   with high persistence lengths, actual bead paths may deviate from ideal Brownian bridges.

2. **Effective Entanglement Density in Defective Networks**:
   When setting a target number of entanglements $N_{\text{ent}}$:

   - With ``filter_dangling_and_soluble=True`` (default), candidate links involving
     dangling chains or sol are discarded, reducing the placed entanglements to
     roughly $w_{\text{active}}^2 N_{\text{ent}}$.
   - With ``filter_dangling_and_soluble=False``, all $N_{\text{ent}}$ links are
     instantiated across the entire volume. Inactive dead-end links carry zero stress
     and are eliminated if :attr:`~pylimer_tools_cpp.StructureSimplificationMode.INACTIVE_THEN_X2F`
     is used.
   - If matching physical experiments where nominal entanglement density is defined relative
     to total volume rather than the gel fraction, users can pass
     ``filter_dangling_and_soluble=False`` or scale target $N_{\text{ent}}$ by
     $1 / w_{\text{active}}^2$.

